# pgvector 노트북 — 2~5교시 (+ 7교시 도전과제)

이 노트북은 `pgvector_심화.md` **2·3·4·5교시**의 실습 가능한 코드를 앞부분에 모으고, 이어서 5교시
"직접 재봅니다"의 원래 실습(및 7교시 도전과제)으로 연결됩니다. 6교시(시연·개인실습)는 `pgvector_2`·
`pgvector_3` 노트북에, 8교시(그래프DB)는 `pgvector_4` 노트북에 따로 있습니다.

```
텍스트
  ↓ [임베딩 모델] (2·3교시)
벡터 (1024차원 숫자 배열)
  ↓ [PostgreSQL + pgvector]
저장 (VECTOR 타입) (3교시)
  ↓ [ANN 인덱스: HNSW or IVFFlat] (4교시 개념 → 5교시 직접 실측)
고속 검색 (가장 가까운 벡터 K개)
  ↓ [메타데이터 필터 결합] (6교시)
정확한 결과
```

**2·3교시 = 왜 필요한가, 어떻게 작동하는가(개념) · 4·5교시 = 안에서 무슨 일이 일어나는가(원리) · 6·7교시 = 직접 구현(실습)**


## 2교시 · 키워드 검색의 한계와 의미 기반 검색

### 모듈 2-1 — 이대로는 못 찾는 문서들

`LIKE '%검색어%'`는 글자가 정확히 일치해야 찾을 수 있습니다. "계약 해지 조건"으로 검색하면
"계약의 종료 및 해제", "서비스 중단 요건", "termination clause"처럼 의미는 같지만 다른 단어로
표현된 문서는 전부 놓칩니다 — 키워드 검색은 글자를 비교하지, 의미를 비교하지 않기 때문입니다.

> 💡 **핵심**: `LIKE` 검색은 텍스트 사전을 보는 것이고, 벡터 검색은 지도에서 가까운 지점을
> 찾는 것입니다. 사전에서 '강아지'를 찾으면 '개'가 바로 옆에 있지 않지만, 지도 위에서는 같은
> 좌표 근방에 놓을 수 있습니다. 단, 벡터 검색도 임베딩 모델이 해당 언어·도메인에 맞게 학습됐을
> 때만 이 거리가 의미를 반영합니다.

> 아래는 참고용 예시이며 **실행하지 않습니다** — 이 `documents` 테이블은 6교시(`pgvector_2`)에서
> 새로 만들며, 지금 이 노트북 환경에는 아직 없습니다.
> ```sql
> SELECT title, metadata ->> 'doc_type' AS 유형
> FROM documents
> WHERE metadata ->> 'content' LIKE '%계약 해지 조건%';
> -- 결과: 0건 (실제로는 관련 문서가 있는데도 다른 단어로 표현돼 못 찾음)
> ```

### 모듈 2-2 — 의미를 좌표로

임베딩이란 텍스트를 고차원 좌표로 변환하는 과정입니다. 같은 의미를 가진 텍스트는 이 좌표
공간에서 가까운 위치에 놓이도록 학습됩니다. 좌표가 있으면 거리를 잴 수 있고, 거리를 잴 수
있으면 "가장 비슷한 것"을 찾을 수 있습니다. 아래에서 실제 모델로 변환해봅니다.


In [ ]:
%pip install -q sentence-transformers pgvector "psycopg[binary]"

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# 💡 이 모델(BAAI/bge-m3)이 이 컴퓨터에 처음 실행되는 경우, 아래 코드가 캐시 유무를 확인해
# 자동으로 온라인 다운로드(최초 1회, 2GB+, 시간이 걸릴 수 있음)를 진행합니다.
# 이미 받아둔 적이 있으면 오프라인 모드로 전환해 인터넷 확인 없이 곧바로 캐시를 씁니다(빠름).
import os
from huggingface_hub import constants as hf_constants

MODEL_NAME = "BAAI/bge-m3"
# 허깅페이스 캐시 폴더 이름 규칙: "models--" + (owner/repo)의 "/"를 "--"로 바꾼 이름
cache_dir_name = "models--" + MODEL_NAME.replace("/", "--")
is_cached = os.path.isdir(os.path.join(hf_constants.HF_HUB_CACHE, cache_dir_name))

if is_cached:
    os.environ["HF_HUB_OFFLINE"] = "1"
    os.environ["TRANSFORMERS_OFFLINE"] = "1"
    print(f"[캐시 발견] {MODEL_NAME} -> 오프라인 모드로 로딩합니다.")
else:
    print(f"[캐시 없음] {MODEL_NAME} -> 온라인으로 최초 1회 다운로드합니다 (인터넷 연결 필요)...")

from sentence_transformers import SentenceTransformer  # 문장 → 숫자 벡터(임베딩)로 바꿔주는 모델 클래스

model = SentenceTransformer(MODEL_NAME)   # 캐시 있으면 5~10초, 없으면 다운로드 시간 추가 — 1024차원 벡터를 만드는 다국어 임베딩 모델

# 의미는 비슷하지만 글자는 다른 문장 3개 + 전혀 다른 주제 문장 1개를 준비합니다.
# (이 4개로 "의미가 비슷하면 벡터도 가까워지는지"를 곧이어 눈으로 확인합니다.)
texts = [
    "계약 해지 조건",
    "계약의 종료 및 해제",
    "서비스 중단 요건",
    "제품 사양서",
]

# model.encode(문장 리스트)는 각 문장을 1024개의 숫자로 이루어진 벡터로 바꿔, 2차원 배열로 돌려줍니다.
vectors = model.encode(texts)
print(vectors.shape)   # (4, 1024) -- 텍스트 4개, 각각 1024차원

[캐시 발견] BAAI/bge-m3 -> 오프라인 모드로 로딩합니다.


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 40129.51it/s]


(4, 1024)


## 3교시 · 임베딩과 거리 지표

### 모듈 3-1 — 임베딩의 정의와 차원

| 용어 | 뜻 |
|---|---|
| 임베딩(embedding) | 텍스트→벡터 변환 결과 |
| 벡터(vector) | 숫자 배열. `[0.23, -0.15, ..., 0.87]` |
| 차원(dimension) | 벡터의 숫자 개수. BGE-M3 = 1024 |
| `VECTOR` 타입 | pgvector의 저장 타입. `VECTOR(1024)` (PostgreSQL 기본 타입 아님) |

> 💡 **핵심**: 벡터는 "텍스트의 지문"입니다. 의미가 비슷한 텍스트의 벡터는 서로 가깝습니다 —
> 단, 지문 판독기(임베딩 모델)가 잘 만들어져 있을 때만 이 관계가 성립합니다.


In [ ]:
# 방금 만든 벡터가 실제로 어떤 모양인지 하나씩 뜯어봅니다.
result = model.encode(["계약 해지 조건", "계약의 종료 및 해제"])
v1, v2 = result[0], result[1]   # 두 문장 각각의 임베딩 벡터를 꺼냅니다.

print(f"차원: {len(v1)}")           # 벡터를 이루는 숫자 개수 (1024가 나와야 정상)
print(f"타입: {v1.dtype}")          # 벡터 안 숫자들의 자료형 (보통 float32 — 32비트 실수)
print(f"첫 5개 값: {v1[:5]}")       # 벡터 앞부분 5개 숫자를 미리보기 (v1[:5] = 슬라이싱, 0~4번째 값)

차원: 1024
타입: float32
첫 5개 값: [-0.03405285 -0.00306596 -0.0321913  -0.01565113  0.00139787]


### 모듈 3-2 — 세 가지 거리 지표

두 벡터 사이 "거리"를 재는 대표적인 방법 셋입니다 — **L2 거리**(작을수록 비슷) · **내적**(클수록
비슷, 방향+크기 반영) · **코사인 유사도**(1에 가까울수록 비슷, 방향만 반영). 세 지표를 직접
계산해보는 것은 바로 아래 "Step 0"에서 합니다.

여기서는 한 가지만 먼저 짚습니다 — **벡터 길이(norm)가 다르면 내적과 코사인이 갈립니다**:


In [ ]:
import numpy as np   # 벡터(숫자 배열) 연산을 위한 라이브러리

# 방향은 완전히 같지만 길이(크기)가 다른 두 벡터를 만들어 비교합니다.
v_long = np.array([3.0, 4.0])    # 길이 = 5.0  (3² + 4² = 25, 제곱근 = 5)
v_short = np.array([0.6, 0.8])   # 길이 = 1.0  (v_long과 방향은 완전히 같음, 길이만 1/5)

# lambda는 이름 없이 한 줄로 정의하는 짧은 함수입니다. cosine(a, b)라고 호출하면 아래 식을 계산합니다.
# np.dot(a, b)      : 내적(dot product) — 두 벡터를 같은 자리끼리 곱해서 다 더한 값
# np.linalg.norm(a) : 벡터 a의 길이(크기)
# 내적을 두 벡터 길이의 곱으로 나누면 "방향이 얼마나 비슷한가"만 남습니다 — 이것이 코사인 유사도입니다.
cosine = lambda a, b: np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print(f"코사인 유사도(방향만): {cosine(v_long, v_short):.4f}")   # 1.0 -- 방향이 같으므로 완전 일치
print(f"내적(방향×크기): {np.dot(v_long, v_short):.4f}")          # 5.0 -- 길이 차이가 반영됨

# 💡 임베딩 모델 대부분은 출력 벡터가 이미 정규화(길이=1)되어 있어, 이 경우 코사인=내적이 됩니다.
# (바로 아래 Step 0에서 정규화 전후를 직접 비교합니다.)

코사인 유사도(방향만): 1.0000
내적(방향×크기): 5.0000


### 모듈 3-3 — pgvector 타입 맛보기 (용어집)

| 용어 | pgvector 표현 | 설명 |
|---|---|---|
| `VECTOR(n)` | 컬럼 타입 | n차원 벡터. 오늘은 `VECTOR(1024)` |
| `<->` | L2 거리 연산자 | 값이 작을수록 가까움 |
| `<#>` | 내적 × -1 연산자 | 값이 작을수록 유사 |
| `<=>` | 코사인 거리 연산자 | `1 - 코사인 유사도`. 값이 작을수록 유사 |
| `ORDER BY ... LIMIT k` | kNN 검색 | 가장 가까운 k개 반환 |

> ⚠️ **흔한 실수**: `<=>`는 코사인 **유사도**가 아니라 코사인 **거리**(`1 - 유사도`)를 반환합니다 —
> 값이 작을수록 유사합니다. 인덱스의 연산자 클래스(`vector_cosine_ops` 등)와 쿼리에서 쓰는
> 연산자(`<=>` 등)가 일치해야 인덱스를 탄다는 점을 6교시 실습에서 확인합니다.


## 4교시 · 완전탐색의 한계와 ANN 인덱스의 내부

### 모듈 4-1 — 완전탐색의 비용

인덱스 없이 벡터 검색을 하면 저장된 모든 벡터와 쿼리 벡터를 비교해야 합니다. 데이터가 100만
건을 넘어가면 한 번의 검색에 수 초가 걸립니다 — 이것이 ANN 인덱스가 필요한 이유입니다. 아래에서
순수 Python으로 먼저 규모를 체감하고, 이어지는 Step 1~6에서 실제 PostgreSQL로 측정합니다.


In [ ]:
import numpy as np
import time   # 코드 실행 시간을 재는 표준 라이브러리

# 실습 규모(1만 건, 1024차원) -- 아래 Step 1의 pgvec_bench(4,000건, 실제 PostgreSQL 테이블)와는
# 별개의, 순수 Python 인메모리 데모입니다.
np.random.seed(42)              # 난수 시드 고정 — 매번 같은 "무작위" 값이 나오게 해 실습 결과를 재현 가능하게 함
n_docs = 10_000                 # 가상의 문서(벡터) 개수
n_dim = 1024                    # 벡터 차원 수 (BGE-M3와 동일하게 맞춤)
corpus = np.random.randn(n_docs, n_dim).astype("float32")   # 문서 1만 개 분량의 무작위 벡터 (n_docs × n_dim 크기 배열)
query = np.random.randn(n_dim).astype("float32")            # 검색에 사용할 무작위 질문 벡터 1개

start = time.perf_counter()     # 정밀한 시각을 기록 — 아래 연산이 끝난 뒤 걸린 시간을 계산하기 위함
# corpus @ query : 행렬 곱셈 연산자(@). corpus의 각 행(문서 벡터)과 query를 내적한 결과를
#                  n_docs개(1만 개) 한 번에 계산합니다. (반복문 없이 numpy가 통째로 빠르게 처리)
# 1 - (내적 / (문서 길이 × 질문 길이)) 는 코사인 거리 공식입니다. 값이 작을수록 더 비슷한 문서.
dists = 1 - (corpus @ query) / (
    np.linalg.norm(corpus, axis=1) * np.linalg.norm(query)
)
top_k = np.argsort(dists)[:10]  # np.argsort로 거리가 작은 순으로 정렬한 인덱스를 얻고, 상위 10개만 선택
elapsed = time.perf_counter() - start   # 시작 시각과의 차이 = 이 연산에 걸린 시간(초)

print(f"{n_docs:,}건 완전탐색: {elapsed * 1000:.1f}ms")   # 초를 밀리초(ms)로 바꿔 출력 (×1000)
# 예상 출력: 10,000건 완전탐색: 약 3~8ms

10,000건 완전탐색: 16.1ms


### 모듈 4-2 — HNSW (계층적 탐색)

여러 층의 그래프를 쌓아 올린 구조입니다. 위 층에서 대략적인 위치를 찾고, 아래층으로 내려오며
점점 정밀하게 좁혀갑니다(고속도로→국도→골목).

| 파라미터 | 기본값 | 의미 |
|---|---|---|
| `m` | 16 | 각 노드가 연결하는 이웃 수 |
| `ef_construction` | 64 | 빌드 시 후보군 크기 |
| `ef_search` | 40 | 검색 시 탐색 폭(세션에서 동적 조정 가능) |

### 모듈 4-3 — IVFFlat (클러스터 분할 후 탐색)

데이터를 여러 클러스터로 나누고, 쿼리와 가까운 클러스터 몇 개만 탐색합니다.

| 항목 | HNSW | IVFFlat |
|---|---|---|
| 정확도 | 높음 | HNSW보다 낮음 |
| 빌드 시간 | 느림 | 빠름 |
| 빌드 전 데이터 | 불필요 | `lists × 39`건 이상 필요 |

> ⚠️ **흔한 실수**: IVFFlat은 인덱스를 만들기 **전에** 데이터가 있어야 합니다 — 빈 테이블에 먼저
> 인덱스를 만들면 클러스터링이 제대로 안 됩니다. (HNSW는 이런 제약이 없습니다.)

실제로 이 두 인덱스를 만들고 recall·속도를 직접 재보는 것은 바로 아래 Step 5·6에서 합니다.

### 5교시를 위한 가설 정리

다음 가설들을 이어지는 5교시에서 직접 검증합니다.

- **가설 1**: 인덱스 없는 검색이 소규모(1만 건)에서는 가장 빠르고 정확하다.
- **가설 2**: HNSW는 `ef_search`를 높일수록 recall이 올라가지만 속도는 느려진다.
- **가설 3**: IVFFlat은 HNSW보다 빌드가 빠르다.
- **가설 4**: 인덱스 없는 검색 대비 HNSW·IVFFlat의 recall은 100%가 아니다.


## 5교시 · 직접 재봅니다

이어서 5교시(직접 재봅니다)·7교시(도전 과제) 실습입니다.

- 실제 pgvector 확장(`db-pg`는 `pgvector/pgvector:pg17` 이미지이므로 이미 설치되어 있음)을 그대로 씁니다.
- 목적: **합성 벡터로 규모(수천~수만 건)를 실험**해 완전탐색의 O(N) 비용과 HNSW·IVFFlat의 recall/속도
  트레이드오프를 직접 측정합니다. (실제 문장의 의미 검색은 6교시의 시연·개인실습 노트북에서 확인합니다.)
- 위 4교시에서 정리한 가설 1~4를 이 노트북의 실측으로 검증합니다.


## 준비 — 접속 및 확장 활성화

In [ ]:
import os, time
import numpy as np
import psycopg                                # PostgreSQL 접속·SQL 실행 드라이버
from dotenv import load_dotenv                # .env 파일 내용을 환경변수로 불러오는 함수
from pgvector.psycopg import register_vector  # numpy 배열 ↔ PostgreSQL vector 타입을 자동 변환해주는 등록 함수

load_dotenv()                                 # 같은 폴더의 .env 파일 → 환경변수로 등록 (여기에 비밀번호가 있음)
pw = os.environ["PGPASSWORD"]                 # 비밀번호는 .env 파일에서 읽어옵니다 (코드에 직접 적지 않음)
# host(주소)·port(포트)·dbname(데이터베이스 이름)을 지정해 PostgreSQL 서버에 접속합니다.
# 접속에 성공하면 conn(연결) 객체가 만들어지고, 이후 이 conn으로 SQL을 실행할 수 있습니다.
conn = psycopg.connect(host="localhost", port=5432, dbname="course_db", user="postgres", password=pw)
print("연결 완료 ->", conn.info.dbname)       # conn.info.dbname으로 실제 접속된 DB 이름을 확인

연결 완료 -> course_db


In [ ]:
# CREATE EXTENSION IF NOT EXISTS vector : PostgreSQL에 pgvector 확장(벡터 타입·거리 연산자)을 설치합니다.
#   "IF NOT EXISTS"가 있어서 이미 설치돼 있으면 에러 없이 그냥 넘어갑니다.
conn.execute("CREATE EXTENSION IF NOT EXISTS vector")
conn.commit()                          # commit()을 호출해야 변경 사항이 실제로 저장(확정)됩니다.
register_vector(conn)   # 확장 활성화 "후"에 호출 -- numpy 배열을 vector 파라미터로 바인딩

# 확장이 잘 켜졌는지 버전을 조회해 확인합니다.
# pg_extension은 설치된 확장 목록을 담은 PostgreSQL 시스템 테이블입니다.
# .fetchone()은 결과 중 첫 번째 행을 가져오고, [0]은 그 행의 첫 번째 값(extversion)을 꺼냅니다.
ver = conn.execute("SELECT extversion FROM pg_extension WHERE extname='vector'").fetchone()[0]
print("pgvector 확장 버전 ->", ver)

pgvector 확장 버전 -> 0.8.6


## Step 0 — 세 거리 지표를 작은 벡터로 손 계산 (3교시 복습)

In [ ]:
# 아주 작은 3차원 벡터 두 개로, 세 가지 거리/유사도 지표를 손으로 계산하듯 확인해봅니다.
a = np.array([1.0, 1.0, 0.0])
b = np.array([1.0, 0.0, 0.0])

# L2 거리(유클리드 거리): 두 점 사이의 "직선 거리". np.linalg.norm(a - b)는 (a-b) 벡터의 길이를 구합니다.
l2 = np.linalg.norm(a - b)
# 내적(dot product): 같은 자리 숫자끼리 곱해서 다 더한 값. 방향과 크기를 모두 반영합니다.
dot = np.dot(a, b)
# 코사인 유사도: 내적을 두 벡터 길이의 곱으로 나눠, 방향(각도)만 비교한 값. 1에 가까울수록 방향이 비슷.
cos_sim = dot / (np.linalg.norm(a) * np.linalg.norm(b))
# 코사인 거리: pgvector의 <=> 연산자가 돌려주는 값과 같은 형태. "유사도"를 "거리"로 뒤집은 것 (1 - 유사도).
cos_dist = 1 - cos_sim

print(f"L2 거리(유클리드): {l2:.4f}")
print(f"내적(dot product): {dot:.4f}  (pgvector의 <#> 는 이 값에 -1을 곱해 '거리'처럼 씀)")
print(f"코사인 유사도: {cos_sim:.4f}  ->  코사인 거리(<=>): {cos_dist:.4f}")

# 정규화(normalize)란 벡터의 "길이"를 1로 맞추고 "방향"만 남기는 것입니다. (벡터 ÷ 자기 길이)
a_norm = a / np.linalg.norm(a)
b_norm = b / np.linalg.norm(b)
# 정규화된 벡터끼리는 내적 = 코사인 유사도가 됩니다 — 길이가 모두 1이라 "크기" 영향이 사라지기 때문입니다.
print(f"정규화 후: dot={np.dot(a_norm, b_norm):.4f}, cosine_sim={cos_sim:.4f}  -- 둘이 같아집니다")

L2 거리(유클리드): 1.0000
내적(dot product): 1.0000  (pgvector의 <#> 는 이 값에 -1을 곱해 '거리'처럼 씀)
코사인 유사도: 0.7071  ->  코사인 거리(<=>): 0.2929
정규화 후: dot=0.7071, cosine_sim=0.7071  -- 둘이 같아집니다


## Step 1 — 실험 전용 테이블 생성 + 클러스터형 합성 벡터 4,000건 적재

`documents`(9/17 만든 4행짜리 표)와는 별도로, 규모를 마음대로 조절할 수 있는 실험 테이블을 만듭니다.
doc_type마다 "중심 벡터"를 만들고 각 문서는 중심 근처에 노이즈를 더해 생성합니다 — 같은 doc_type끼리는
벡터가 가깝고, 다른 doc_type과는 멀어지도록 하여 실제 임베딩의 군집 성질을 흉내 냅니다.

In [ ]:
# 실험 전용 테이블을 새로 만듭니다. DROP TABLE IF EXISTS로 먼저 지워서, 재실행해도 항상 깨끗하게 시작합니다.
conn.execute("DROP TABLE IF EXISTS pgvec_bench")
# 아래 테이블은 컬럼 3개로 이루어집니다.
#   id        BIGSERIAL PRIMARY KEY : 자동으로 1씩 늘어나는 고유 번호(기본키)
#   doc_type  TEXT                 : 문서 종류(계약서/재무제표 등) — 나중에 필터 검색에 사용
#   embedding VECTOR(1024)         : pgvector 확장이 제공하는 벡터 타입, 1024차원
conn.execute('''
CREATE TABLE pgvec_bench (
    id BIGSERIAL PRIMARY KEY,
    doc_type TEXT,
    embedding VECTOR(1024)
)
''')
conn.commit()

# 아래에서 "클러스터형" 합성 벡터를 직접 만듭니다. 실제 임베딩 모델 없이도,
# "같은 종류의 문서는 벡터 공간에서 가까이 모여 있다"는 상황을 흉내내기 위한 방법입니다.
rng = np.random.default_rng(42)      # 이 셀 전용 난수 생성기 (시드 고정 — 매번 같은 결과 재현)
DIM, N_DOCS = 1024, 4000             # 벡터 차원 수, 만들 문서 개수
DOC_TYPES = ["계약서", "재무제표", "인사규정", "기술사양서", "특허명세서"]   # 5가지 문서 종류

# 문서 종류마다 "중심점" 벡터를 하나씩 무작위로 만듭니다. (5개 종류 × 1024차원)
centers = rng.normal(size=(len(DOC_TYPES), DIM)).astype(np.float32)
centers /= np.linalg.norm(centers, axis=1, keepdims=True)   # 각 중심점을 길이 1로 정규화

# 4,000개 문서 각각에 무작위로 문서 종류(0~4번)를 배정합니다.
doc_type_idx = rng.integers(0, len(DOC_TYPES), size=N_DOCS)
# 각 문서의 벡터 = 자기 종류의 중심점 + 약간의 무작위 잡음(noise, scale=0.3)
# → 같은 종류끼리는 중심점 근처에 모이되, 완전히 똑같지는 않게(잡음으로 흩뿌려서) 만듭니다.
embeddings = centers[doc_type_idx] + rng.normal(scale=0.3, size=(N_DOCS, DIM)).astype(np.float32)
embeddings /= np.linalg.norm(embeddings, axis=1, keepdims=True)   # 최종 벡터도 길이 1로 정규화

# with conn.cursor() as cur: 로 커서(SQL 실행 창구)를 열고, executemany로 4,000건을 한 번에 삽입합니다.
# (executemany는 같은 SQL을 여러 값 세트에 반복 적용 — 한 건씩 execute()하는 것보다 훨씬 빠릅니다.)
with conn.cursor() as cur:
    cur.executemany(
        "INSERT INTO pgvec_bench (doc_type, embedding) VALUES (%s, %s)",
        [(DOC_TYPES[doc_type_idx[i]], embeddings[i]) for i in range(N_DOCS)]
    )
conn.commit()

cnt = conn.execute("SELECT COUNT(*) FROM pgvec_bench").fetchone()[0]
print(f"생성된 문서: {cnt}건")

생성된 문서: 4000건


## Step 2 — 최근접 6건 검색 (브루트포스, 인덱스 없음)

In [ ]:
# 방금 만든 4,000건 중 첫 번째 문서(embeddings[0])를 질문 벡터로 삼아, 가장 가까운 6건을 찾습니다.
# 아직 인덱스를 만들지 않았으므로, PostgreSQL은 4,000건 전체를 하나씩 다 비교하는 "브루트포스(완전탐색)"로 검색합니다.
query = embeddings[0]
result = conn.execute('''
    SELECT id, doc_type, embedding <=> %s AS distance
    FROM pgvec_bench ORDER BY embedding <=> %s LIMIT 6
''', (query, query)).fetchall()
# <=> 는 pgvector의 코사인 거리 연산자입니다. 값이 작을수록 두 벡터가 더 비슷합니다.
# ORDER BY ... LIMIT 6 으로, 거리가 가장 작은(가장 비슷한) 6건만 가져옵니다.
# %s가 두 번 나오지만 같은 query 벡터를 SELECT용과 ORDER BY용으로 각각 넘겨준 것입니다.

for r in result:
    # r[0]=id, r[1]=doc_type, r[2]=distance 순서로 튜플 안에 들어있습니다.
    print(f"  id={r[0]:4d}  doc_type={r[1]:8s}  distance={r[2]:.4f}")

# 1위는 자기 자신(질문으로 쓴 문서 그 자체)이라 거리가 항상 0입니다. 그다음 상위 5건 중에서,
# 질문 문서와 "같은 doc_type"인 것이 몇 개인지 비율로 계산합니다 — 클러스터가 잘 나뉘어 있다면 높게 나옵니다.
same_type = sum(1 for r in result[1:] if r[1] == DOC_TYPES[doc_type_idx[0]]) / 5
print(f"\n1위(자기 자신, 거리 0) 제외 상위 5건 중 같은 종류 비율: {same_type:.0%}")

  id=   1  doc_type=특허명세서     distance=0.0000
  id= 643  doc_type=특허명세서     distance=0.8757
  id= 199  doc_type=기술사양서     distance=0.8954
  id=1862  doc_type=인사규정      distance=0.8967
  id= 229  doc_type=재무제표      distance=0.9010
  id= 496  doc_type=재무제표      distance=0.9036

1위(자기 자신, 거리 0) 제외 상위 5건 중 같은 종류 비율: 20%


## Step 3 — 메타데이터 필터 + 벡터 검색 결합

In [ ]:
# "메타데이터 필터"와 "벡터 검색"을 하나의 함수로 결합합니다.
# doc_type이 주어지면 먼저 그 종류로 범위를 좁힌 뒤(WHERE) 벡터 거리로 순위를 매기고,
# 주어지지 않으면 전체 문서를 대상으로 벡터 검색만 합니다.
def hybrid_search(query_vec, doc_type=None, k=5):
    if doc_type is not None:
        return conn.execute(
            # WHERE doc_type = %s 로 먼저 범위를 좁히고, 그 안에서만 <=>(코사인 거리) 순으로 정렬합니다.
            "SELECT id, doc_type, embedding <=> %s AS distance FROM pgvec_bench "
            "WHERE doc_type = %s ORDER BY embedding <=> %s LIMIT %s",
            (query_vec, doc_type, query_vec, k)
        ).fetchall()
    # doc_type이 None(지정 안 함)이면 필터 없이 전체 문서 대상으로 검색합니다.
    return conn.execute(
        "SELECT id, doc_type, embedding <=> %s AS distance FROM pgvec_bench "
        "ORDER BY embedding <=> %s LIMIT %s", (query_vec, query_vec, k)
    ).fetchall()

# doc_type="계약서"로 좁혀서 검색해봅니다.
rows = hybrid_search(query, doc_type="계약서", k=5)
print("doc_type='계약서'로 좁힌 뒤 벡터 검색:")
for r in rows:
    print(" ", r)

doc_type='계약서'로 좁힌 뒤 벡터 검색:
  (3959, '계약서', 0.9114141954677332)
  (3819, '계약서', 0.9212800794148586)
  (3491, '계약서', 0.9252833389167989)
  (2062, '계약서', 0.9254349828284953)
  (754, '계약서', 0.9287512771042483)


## Step 4 — 문서 수 증가에 따른 브루트포스 검색 시간 실측

In [ ]:
# 문서 수가 늘어날수록 인덱스 없는 브루트포스 검색이 얼마나 느려지는지 실측합니다.
# id <= target_n 조건으로 "문서가 이만큼만 있다고 치면" 상황을 흉내내는 방식입니다.
for target_n in [500, 1000, 2000, 4000]:
    times = []
    for _ in range(5):   # 한 번만 재면 우연히 튈 수 있으니, 5번 반복해 평균·중앙값을 봅니다.
        t0 = time.perf_counter()               # 측정 시작 시각
        conn.execute(
            "SELECT id FROM pgvec_bench WHERE id <= %s ORDER BY embedding <=> %s LIMIT 10",
            (target_n, query)
        ).fetchall()
        times.append((time.perf_counter() - t0) * 1000)   # 걸린 시간(초)을 밀리초로 변환해 기록
    # np.mean(리스트) = 평균, np.median(리스트) = 중앙값(가운데 값 — 극단값에 덜 흔들림)
    print(f"문서 수 {target_n:5d}건 -> 평균 {np.mean(times):.2f}ms (중앙값 {np.median(times):.2f}ms)")

문서 수   500건 -> 평균 2.26ms (중앙값 2.25ms)
문서 수  1000건 -> 평균 3.19ms (중앙값 3.13ms)
문서 수  2000건 -> 평균 4.64ms (중앙값 4.60ms)
문서 수  4000건 -> 평균 9.06ms (중앙값 8.92ms)


## Step 5 — HNSW 인덱스 생성 + recall/속도 실측

먼저 20개의 쿼리에 대해 브루트포스로 "정답"(top-10)을 구해두고, HNSW 인덱스를 만든 뒤 같은 쿼리로
recall과 지연시간을 비교합니다.

In [ ]:
# HNSW/IVFFlat 인덱스의 "recall(재현율)"을 재려면, 먼저 "진짜 정답"이 뭔지 알아야 합니다.
# 그래서 인덱스 없이 브루트포스(완전탐색)로 20개 질문의 정답 top-10을 미리 구해둡니다.
# (인덱스는 속도를 위해 정확도를 조금 희생하는 방법이라, 이 "정답"과 비교해야 얼마나 희생했는지 알 수 있습니다.)
n_queries = 20
query_ids = rng.choice(N_DOCS, size=n_queries, replace=False)   # 4,000건 중 중복 없이 20개를 질문으로 무작위 선택
gt_sets, bf_times = [], []   # gt = ground truth(정답), bf = brute force(브루트포스)
for qi in query_ids:
    qv = embeddings[qi]                         # 선택된 문서의 벡터를 질문으로 사용
    t0 = time.perf_counter()
    rows = conn.execute("SELECT id FROM pgvec_bench ORDER BY embedding <=> %s LIMIT 10", (qv,)).fetchall()
    bf_times.append((time.perf_counter() - t0) * 1000)
    gt_sets.append({r[0] for r in rows})        # {r[0] for r in rows} = 결과 id들을 집합(set)으로 저장 (나중에 겹치는 개수 비교용)
print(f"브루트포스(정답) 평균 지연시간: {np.mean(bf_times):.2f}ms")

브루트포스(정답) 평균 지연시간: 9.09ms


In [ ]:
# 이 함수는 인덱스를 하나 만들고(또는 설정만 바꾸고) recall·속도·인덱스 생성 시간을 재서 한 줄로 출력합니다.
# label: 결과에 표시할 이름 / setup_sql: 인덱스를 새로 만드는 SQL(없으면 None) / session_sql: SET으로 파라미터만 바꾸는 SQL
def bench(label, setup_sql, session_sql):
    build_ms = None
    if setup_sql:                                # 인덱스를 새로 만드는 경우에만 생성 시간을 측정
        t0 = time.perf_counter()
        conn.execute(setup_sql)
        conn.commit()
        build_ms = (time.perf_counter() - t0) * 1000
    if session_sql:                              # ef_search, probes 같은 검색 파라미터를 세션 단위로 설정
        conn.execute(session_sql)

    recalls, lat = [], []
    for qi, gt in zip(query_ids, gt_sets):        # 앞서 만든 20개 질문 + 각각의 정답(gt)을 순서대로 짝지어 순회
        qv = embeddings[qi]
        t0 = time.perf_counter()
        rows = conn.execute("SELECT id FROM pgvec_bench ORDER BY embedding <=> %s LIMIT 10", (qv,)).fetchall()
        lat.append((time.perf_counter() - t0) * 1000)   # 이번 질문의 검색 소요 시간(ms)
        got = {r[0] for r in rows}                       # 인덱스를 사용한 검색 결과
        # recall = (인덱스 결과와 정답이 겹치는 개수) / 10. 1.0이면 정답을 100% 그대로 찾았다는 뜻.
        # got & gt 는 두 집합(set)의 교집합 — 공통으로 들어있는 id만 남깁니다.
        recalls.append(len(got & gt) / 10)

    b = f"{build_ms:.0f}ms" if build_ms is not None else "(동일 인덱스)"
    print(f"{label:32s} 평균 {np.mean(lat):6.2f}ms  recall={np.mean(recalls):.2f}  빌드 {b}")

# HNSW 인덱스를 새로 만들고(m=16, ef_construction=64), ef_search=40으로 검색 정확도를 실측합니다.
#   m, ef_construction : 인덱스를 만들 때(빌드 타임)의 정교함을 조절하는 파라미터
#   ef_search          : 검색할 때(런타임) 얼마나 넓게 후보를 탐색할지 조절 — 클수록 정확하지만 느려짐
bench("HNSW (m=16, ef_construction=64, ef_search=40)",
      "CREATE INDEX ON pgvec_bench USING hnsw (embedding vector_cosine_ops) WITH (m=16, ef_construction=64)",
      "SET hnsw.ef_search = 40")
# 인덱스는 그대로 두고(setup_sql=None) ef_search만 100으로 올려서 recall·속도가 어떻게 바뀌는지 비교합니다.
bench("HNSW (ef_search=100)", None, "SET hnsw.ef_search = 100")

HNSW (m=16, ef_construction=64, ef_search=40) 평균   1.36ms  recall=0.78  빌드 2149ms
HNSW (ef_search=100)             평균   2.03ms  recall=0.88  빌드 (동일 인덱스)


In [ ]:
bench("HNSW (ef_search=70)", None, "SET hnsw.ef_search = 70")

HNSW (ef_search=70)              평균   2.44ms  recall=0.84  빌드 (동일 인덱스)


## Step 6 — IVFFlat 인덱스 생성 + recall/속도 실측

In [ ]:
# 테이블 하나에는 인덱스를 동시에 여러 종류 두면 헷갈리므로, HNSW 인덱스를 지우고 이번엔 IVFFlat을 시험합니다.
conn.execute("DROP INDEX IF EXISTS pgvec_bench_embedding_idx")
conn.commit()

# IVFFlat 인덱스를 새로 만들고(lists=50 — 벡터들을 50개 그룹으로 미리 나눠둠), probes=2로 검색합니다.
#   lists  : 인덱스를 만들 때 전체 벡터를 몇 개의 그룹(리스트)으로 나눌지
#   probes : 검색할 때 그 그룹 중 몇 개까지 들여다볼지 — 클수록 정확하지만 느려짐 (HNSW의 ef_search와 비슷한 역할)
bench("IVFFlat (lists=50, probes=2)",
      "CREATE INDEX ON pgvec_bench USING ivfflat (embedding vector_cosine_ops) WITH (lists=50)",
      "SET ivfflat.probes = 2")
# 인덱스는 그대로 두고 probes만 10으로 올려 recall·속도 변화를 비교합니다.
bench("IVFFlat (probes=10)", None, "SET ivfflat.probes = 10")

IVFFlat (lists=50, probes=2)     평균   0.72ms  recall=0.18  빌드 226ms
IVFFlat (probes=10)              평균   1.11ms  recall=0.40  빌드 (동일 인덱스)


In [ ]:
# 값 바꿔보기
conn.execute("DROP INDEX IF EXISTS pgvec_bench_embedding_idx")
conn.commit()

# IVFFlat 인덱스를 새로 만들고(lists=50 — 벡터들을 50개 그룹으로 미리 나눠둠), probes=2로 검색합니다.
#   lists  : 인덱스를 만들 때 전체 벡터를 몇 개의 그룹(리스트)으로 나눌지
#   probes : 검색할 때 그 그룹 중 몇 개까지 들여다볼지 — 클수록 정확하지만 느려짐 (HNSW의 ef_search와 비슷한 역할)
bench("IVFFlat (lists=30, probes=3)",
      "CREATE INDEX ON pgvec_bench USING ivfflat (embedding vector_cosine_ops) WITH (lists=30)",
      "SET ivfflat.probes = 3")
# 인덱스는 그대로 두고 probes만 10으로 올려 recall·속도 변화를 비교합니다.
bench("IVFFlat (probes=6)", None, "SET ivfflat.probes = 6")

IVFFlat (lists=30, probes=3)     평균   0.83ms  recall=0.30  빌드 317ms
IVFFlat (probes=6)               평균   0.89ms  recall=0.42  빌드 (동일 인덱스)


> **가설 검증**: 위 결과표(직접 실행한 값)를 `pgvector_심화.md` 5교시 모듈 5-2의 실측표와 비교해보세요.
> 소규모(4,000건)에서는 HNSW와 브루트포스가 recall·속도 모두 비슷하고, IVFFlat은 훨씬 빠르지만
> recall이 낮게 나오는 것을 직접 확인할 수 있습니다.

## 🔰 미션 — 파라미터를 바꿔가며 recall 곡선 그려보기 (7교시 도전 1과 연결)

> ⚠️ **흔한 실수 (실제로 이 노트북을 만들며 겪은 실수입니다)**: Step 6에서 `DROP INDEX` 후
> IVFFlat을 만들었기 때문에, 지금 `pgvec_bench`에는 **HNSW가 아니라 IVFFlat 인덱스**가 걸려
> 있습니다. 이 상태에서 `SET hnsw.ef_search`만 바꾸면 recall이 전혀 변하지 않습니다(해당 세션
> 파라미터가 적용될 HNSW 인덱스 자체가 없기 때문입니다) — 실제로 처음 이 노트북을 작성했을 때
> ef_search를 10~100까지 바꿔도 recall이 0.38로 고정되는 것을 보고서야 이 사실을 발견했습니다.
> **`SET`으로 파라미터를 조정하기 전에 해당 인덱스 타입이 실제로 걸려 있는지 먼저 확인하세요.**
> 아래 셀에서 HNSW를 다시 만든 뒤 스윕합니다.

In [ ]:
# IVFFlat을 지우고 HNSW를 다시 만든 뒤에 스윕해야 ef_search가 실제로 효과를 냅니다.
# (바로 위에서 IVFFlat 인덱스로 바꿔뒀기 때문에, 지금 그대로 ef_search를 바꾸면 IVFFlat엔 없는
#  파라미터라 아무 효과가 없습니다 — 반드시 HNSW 인덱스가 걸려 있어야 ef_search 스윕이 의미가 있습니다.)
conn.execute("DROP INDEX IF EXISTS pgvec_bench_embedding_idx")
conn.commit()
conn.execute("CREATE INDEX ON pgvec_bench USING hnsw (embedding vector_cosine_ops) WITH (m=16, ef_construction=64)")
conn.commit()

# "스윕(sweep)"이란 한 파라미터 값을 여러 개로 바꿔가며 결과가 어떻게 달라지는지 쭉 훑어보는 것입니다.
# 아래 ef_values를 바꿔가며 recall·지연시간이 어떻게 변하는지 직접 실행해보세요.
ef_values = [10, 20, 40, 80, 100]
print("ef_search 스윕 (pgvec_bench, HNSW 인덱스를 방금 다시 생성한 상태)")
for ef in ef_values:
    # setup_sql=None 이므로 인덱스는 그대로 두고, SET으로 ef_search 값만 바꿔가며 매번 재측정합니다.
    bench(f"HNSW (ef_search={ef})", None, f"SET hnsw.ef_search = {ef}")

ef_search 스윕 (pgvec_bench, HNSW 인덱스를 방금 다시 생성한 상태)
HNSW (ef_search=10)              평균   0.80ms  recall=0.75  빌드 (동일 인덱스)
HNSW (ef_search=20)              평균   0.88ms  recall=0.75  빌드 (동일 인덱스)
HNSW (ef_search=40)              평균   1.29ms  recall=0.76  빌드 (동일 인덱스)
HNSW (ef_search=80)              평균   1.77ms  recall=0.83  빌드 (동일 인덱스)
HNSW (ef_search=100)             평균   2.02ms  recall=0.86  빌드 (동일 인덱스)


## 보너스 — documents 테이블에 실제로 적용한다면 (개념 확인)

9/17 만든 `documents`(4행)에 실제로 `embedding VECTOR(1024)` 컬럼을 추가하는 절차를 확인합니다.
여기서는 BGE-M3 대신 **일부러 눈에 띄는 작은 더미 벡터**를 채워 SQL 절차 자체(ALTER TABLE ->
UPDATE -> 검색)가 문제없이 동작하는지만 확인합니다 — 실제 의미 있는 임베딩은 6교시 시연·개인실습
노트북의 BGE-M3 결과를 참고하세요.

In [ ]:
# 개념 확인용 — 9/17에 만들어둔 실제 documents 테이블에 embedding 컬럼을 "잠깐" 추가해봅니다.
# ALTER TABLE ... ADD COLUMN : 기존 테이블에 새 컬럼(열)을 추가하는 SQL. IF NOT EXISTS로 재실행 안전.
conn.execute("ALTER TABLE documents ADD COLUMN IF NOT EXISTS embedding vector(1024)")
conn.commit()

# 더미 벡터로 채워 절차만 확인 (실제 서비스에서는 BGE-M3 encode() 결과를 사용)
# documents 테이블에 있는 모든 id를 순서대로 가져옵니다.
doc_ids = [r[0] for r in conn.execute("SELECT id FROM documents ORDER BY id").fetchall()]
rng2 = np.random.default_rng(0)   # 이 셀 전용 난수 생성기 (다른 셀의 rng와 섞이지 않도록 별도 이름 사용)
for did in doc_ids:
    dummy = rng2.normal(size=1024).astype(np.float32)   # 진짜 임베딩 대신, 절차 확인용 무작위 1024차원 벡터
    dummy /= np.linalg.norm(dummy)                       # 길이 1로 정규화 (실제 임베딩 모델 출력과 형태를 맞춤)
    # UPDATE ... SET ... WHERE id = %s : 각 문서 행을 하나씩 찾아 embedding 값을 채워 넣습니다.
    conn.execute("UPDATE documents SET embedding = %s WHERE id = %s", (dummy, did))
conn.commit()

# IS NOT NULL 조건으로 embedding이 채워진 행이 몇 개인지 세어 확인합니다.
cnt = conn.execute("SELECT count(*) FROM documents WHERE embedding IS NOT NULL").fetchone()[0]
print(f"documents.embedding 채움 확인: {cnt}건 (더미 벡터 -- SQL 절차 검증용)")

# 되돌리기: 이 컬럼은 개념 확인용이므로 실습 정리 차원에서 제거합니다.
# ALTER TABLE ... DROP COLUMN : 방금 추가한 embedding 컬럼을 통째로 삭제해, documents를 원래 상태로 되돌립니다.
conn.execute("ALTER TABLE documents DROP COLUMN IF EXISTS embedding")
conn.commit()
print("정리 완료 -- documents는 9/17 상태(embedding 없음)로 복원했습니다.")

UndefinedTable: relation "documents" does not exist

## 정리

```python
conn.execute("DROP TABLE IF EXISTS pgvec_bench")
conn.commit()
conn.close()
```

실습이 끝나면 실험 테이블을 정리하고 연결을 닫습니다. `documents`는 위에서 이미 원상 복구했습니다.

In [ ]:
# 실험용으로 만들었던 pgvec_bench 테이블을 지우고, 데이터베이스 연결도 정리(종료)합니다.
conn.execute("DROP TABLE IF EXISTS pgvec_bench")
conn.commit()
conn.close()   # 연결을 닫아 자원을 반납합니다 — 이후 이 conn으로는 더 이상 SQL을 실행할 수 없습니다.
print("정리 완료")